In [1]:
from cornac.data import Reader
from cornac.datasets import movielens
from cornac.data import Dataset, FeatureModality
from cornac.eval_methods import RatioSplit, StratifiedSplit
from cornac.metrics import RMSE
from cornac.models import MF, ItemKNN, UserKNN, NMF, BPR
import pandas as pd
import numpy as np
import cornac
import math
import seaborn as sns
import matplotlib.pyplot as plt


/Users/tahsinalamgirkheya/anaconda3/envs/cornac/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current working directory: /Users/tahsinalamgirkheya/Desktop/work/cornac-master
Files in current directory: ['.DS_Store', 'kheya_test.ipynb', '.codecov.yml', 'pytest.ini', 'LICENSE', 'requirements.txt', 'AUTHORS', 'Dockerfile', 'tests', 'MANIFEST.in', '.coveragerc', 'docs', '.readthedocs.yml', 'README.md', 'setup.py', '.gitignore', 'examples', 'cornac.egg-info', '.github', 'tutorials', 'setup.cfg', 'docker-compose.yml', 'build', 'cornac', '.git', '.circleci', 'flow.jpg']
Files in './cornac/data': ['.DS_Store', 'indexed_movies.csv', 'indexed_interactions.csv', 'u_id_mapping.csv']


In [15]:
reader = Reader()
rating_data = pd.read_csv(
    "./cornac/data/indexed_interactions.csv",
    sep="\t",
    header=None,
    names=["userID", "itemID", "Rating", "Timestamp"],
)
# rating_data = rating_data.drop(columns=rating_data.columns[-1])
rating_data = rating_data.to_numpy()
rating_data
# user id itemid rating and timestamp

array([[        0,         0,         5, 978300760],
       [        0,         1,         3, 978302109],
       [        0,         2,         3, 978301968],
       ...,
       [     6039,       365,         5, 956704746],
       [     6039,       152,         4, 956715648],
       [     6039,        26,         4, 956715569]])

In [18]:
# movie_data = reader.read(fpath="./data/indexed_movies.csv", sep=",", fmt="UIRT")
# movie_data
movies = pd.read_csv("./cornac/data/indexed_movies.csv")

movies = movies.drop(columns=movies.columns[0])
movies[:4]

unique_genres = set("|".join(movies["genres"]).split("|"))
unique_genres = list(unique_genres)

for genre in unique_genres:
    movies[genre] = 0
for index, row in movies.iterrows():
    genres = row["genres"].split("|")
    for genre in genres:
        movies.at[index, genre] = 1

# item_categories = movies[unique_genres]
genre = movies[unique_genres]
item_features_numpy = genre.to_numpy()
# print(item_features_numpy.shape)

# item_categories = item_categor
item_features = {
    str(item_id): {"genre_" + str(idx): value for idx, value in enumerate(row)}
    for item_id, row in enumerate(item_features_numpy)
}
ids = list(range(0, 3416))
item_feature_modality = FeatureModality(
    features=item_features_numpy, ids=ids, normalized=True
)
item_feature_modality.__getstate__()
users = pd.read_csv("./cornac/data/u_id_mapping.csv", sep="\t")
users = users.drop(columns=users.columns[0])
gender_map = {"M": 0, "F": 1}
users["Gender"] = users["Gender"].map(gender_map)

user_features_numpy = users.to_numpy()
print(user_features_numpy.shape)
user_feature_modality = FeatureModality(
    features=user_features_numpy, name="user", normalized=True, ids=list(range(0, 6040))
)
dataset = rating_data
# print("Example Item Features:")
# for item_id, features in list(item_features.items())[:5]:
#     print(f"Item ID: {item_id}, Features: {features}")

(6040, 2)


In [20]:
ratio_split = StratifiedSplit(
    data=dataset, test_size=0.2, rating_threshold=0.0, seed=123, verbose=True
)

model = MF(
    k=10, max_iter=50, learning_rate=0.01, lambda_reg=0.02, seed=123, name="lmd0.02"
)

cornac.Experiment(
    ratio_split, models=[model], metrics=[cornac.metrics.RMSE(),cornac.metrics.AUC()]
).run()

rating_threshold = 0.0
exclude_unknowns = True
---
Training data:
Number of users = 6040
Number of items = 3416
Number of ratings = 797275
Max rating = 5.0
Min rating = 1.0
Global mean = 3.6
---
Test data:
Number of users = 6040
Number of items = 3416
Number of ratings = 202336
Number of unknown users = 0
Number of unknown items = 0
---
Total users = 6040
Total items = 3416

[lmd0.02] Training started!
heyaa

[lmd0.02] Evaluation started!


Ranking: 100%|██████████| 6040/6040 [00:02<00:00, 2285.57it/s]


TEST:
...
        |   RMSE |    AUC | Train (s) | Test (s)
------- + ------ + ------ + --------- + --------
lmd0.02 | 0.8611 | 0.6934 |    1.2063 |   3.7410

